In [ ]:
import rebound
import numpy as np
import matplotlib.pyplot as plt
from rebound import hash as h

MONITOR SIMULATION AS IT RUNS

In [ ]:
# Import simulation archive
sa = rebound.Simulationarchive("outputs/orbital_integration/archive.bin")

print("Number of snapshots: {}".format(len(sa)))
print("Time of first and last snapshot: {:.1f}, {:.1f}".format(sa.tmin, sa.tmax))

N_pl = 8 # Total number of bodies
N_tp = 15 # Number of test particles

In [ ]:
# Access each snapshot in the simulation

# Earliest snapshot
sim = sa[0]
rebound.OrbitPlot(sim, unitlabel = "[AU]", xlim = [-5, 5], ylim = [-6, 6],
                  color = (N_pl - 1) * ["black"] + N_tp * ["red"])

In [ ]:
# Latest snapshot
sim = sa[-1]
rebound.OrbitPlot(sim, unitlabel = "[AU]", xlim = [-5, 5], ylim = [-6, 6],
                  color = (N_pl - 1) * ["black"] + N_tp * ["red"])
sim.status()

In [ ]:
# Convert values into instantaneous orbital elements
# AKA Osculating Orbital Elements
orbits = sim.orbits()
for orbit in orbits:
    print(orbit)

EXTRACT INFORMATION ABOUT A SINGLE PARTICLE

In [ ]:
# Extract info about a single particle
t = np.zeros(len(sa))
a = np.zeros(len(sa))
e = np.zeros(len(sa))

pid = 100 # Test particle ID
for i, sim in enumerate(sa):
    t[i] = sim.t / 1e6
    try:
        a[i] = sim.particles[h(pid)].a
        e[i] = sim.particles[h(pid)].e
    except rebound.ParticleNotFound: # Particle ejected
        a = a[:i]
        e = e[:i]
        t = t[:i]
        break

# Plot
plt.plot(t, a, label = "Semimajor axis")
plt.xlabel("Time (Myr)")
plt.ylabel("a (AU)")
plt.title("Particle {0}".format(pid))
plt.legend()

In [ ]:
# Add perehelion distance (q), aphelion distance (Q) to plot
q = a * (1 - e)
Q = a * (1 + e)

plt.plot(t, a, label = "Semimajor axis")
plt.plot(t, q, label = "Perehelion distance")
plt.plot(t, Q, label = "Aphelion distance")
plt.xlabel("Time (Myr)")
plt.ylabel("Value (AU)")
plt.title("Particle {0}".format(pid))
plt.legend()

MAKE A MOVIE OF PARTICLE'S ORBIT

In [ ]:
# Make movie of orbit's evolution from top-down perspective

pid = 104

# op1 = rebound.OrbitPlot(sa[0], orbit_style = "solid", 
#                         lw = 1, particles = [1, 2, 3, 4, 5, 6])
# op1.particles.set_sizes([0])

# op2 = rebound.OrbitPlot(sa[0], fig = op1.fig, ax = op1.ax, 
#                         orbit_style = "solid", lw = 1, 
#                         particles = [h(pid)], color = "red")
# op2.particles.set_sizes([0])

count = 0 # Number frames consecutively
fig, ax = plt.subplots()

for i, sim in enumerate(sa):
    if i % 100: # Only plot every 100th frame
        continue
    try:
        op1 = rebound.OrbitPlot(sim, fig = fig, ax = ax, orbit_style = "solid",
                                lw = 1, particles = [1, 2, 3, 4, 5, 6])
        op1.particles.set_sizes([0])

        op2 = rebound.OrbitPlot(sa[0], fig = op1.fig, ax = op1.ax, 
                                orbit_style = "solid", lw = 1, 
                                particles = [h(pid)], color = "red")
        op2.particles.set_sizes([0])

        ax.set_aspect("equal")
        ax.set_xlim(-8, 8)
        ax.set_ylim(-8, 8)
        ax.set_xlabel("x [AU]")
        ax.set_ylabel("y [AU]")

        ax.set_title("Particle {0}".format(pid))
        ax.text(-4, -7, "t = {:4.1f} Myr".format(sim.t / 1e6))
        fig.savefig("outputs/orbital_integration/movie/frame_{:04d}.png".format(count))
        
        count += 1
        ax.clear()

    except rebound.ParticleNotFound:
        break